https://www.freesurfer.net/fswiki/CorticalParcellation_Yeo2011

https://www.freesurfer.net/fswiki/CerebellumParcellation_Buckner2011

In [1]:
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn.image import resample_to_img, load_img
from nilearn.datasets import fetch_atlas_harvard_oxford, fetch_atlas_yeo_2011
from scipy.ndimage import label, generic_filter
import xml.etree.ElementTree as ET

In [2]:
# === Input Z map ===
# ============================================== !!!!! CHOOSE THE APPROPRIATE MAP !!!!! ==============================================
zmap_path = "/Users/X/PSILAUT/wDC/Results/masked_group1_original-z-score-map.nii.gz"
# ============================================== !!!!! CHOOSE THE APPROPRIATE MAP !!!!! ==============================================
zmap_img = load_img(zmap_path)
zmap_data = zmap_img.get_fdata()
zmap_affine = zmap_img.affine

# === Threshold Z map ===
z_thresh = 0
binary_mask = (zmap_data > z_thresh).astype(int)
structure = np.ones((3, 3, 3), dtype=int)
labeled_array, num_clusters = label(binary_mask, structure=structure)

# === Load Buckner atlas + LUT ===
buckner_path = "/Users/X/PSILAUT/wDC/Atlas/Buckner_JNeurophysiol11_MNI152/Buckner2011_7Networks_MNI152_FreeSurferConformed1mm_LooseMask.nii.gz"
buckner_lut_path = "/Users/X/PSILAUT/wDC/Atlas/Buckner_JNeurophysiol11_MNI152/Buckner2011_7Networks_ColorLUT.txt"
buckner_labels = {}

with open(buckner_lut_path, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2 and parts[0].isdigit():
            idx = int(parts[0])
            label_end = next((i for i, p in enumerate(parts[1:], 1) if p.isdigit()), len(parts))
            buckner_labels[idx] = " ".join(parts[1:label_end])

# === Map Buckner labels like "7Networks_6" → "Frontoparietal" ===
yeo7_map = {
    "7Networks_1": "Visual", "7Networks_2": "Somatomotor", "7Networks_3": "Dorsal Attention",
    "7Networks_4": "Ventral Attention", "7Networks_5": "Limbic", "7Networks_6": "Frontoparietal",
    "7Networks_7": "Default"
}
for key, val in list(buckner_labels.items()):
    if val in yeo7_map:
        buckner_labels[key] = yeo7_map[val]

# === Load FNIRT cerebellar atlas and parse XML LUT ===
cereb_path = "/Users/X/PSILAUT/wDC/Atlas/Cerebellum-MNIfnirt-maxprob-thr25-2mm.nii.gz"
cereb_lut_path = "/Users/X/PSILAUT/wDC/Atlas/Cerebellum_MNIfnirt.xml"
cereb_labels = {}

tree = ET.parse(cereb_lut_path)
root = tree.getroot()
for label in root.iter('label'):
    idx = int(label.attrib['index'])
    name = label.text.strip()
    cereb_labels[idx] = name

# === Fetch and load Harvard-Oxford + Yeo atlases ===
cortical = fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
subcortical = fetch_atlas_harvard_oxford('sub-maxprob-thr25-2mm')
yeo7_img = load_img(fetch_atlas_yeo_2011()['thick_7'])

# === Resample everything to zmap ===
cort_img = resample_to_img(load_img(cortical.maps), zmap_img, interpolation='nearest')
sub_img = resample_to_img(load_img(subcortical.maps), zmap_img, interpolation='nearest')
yeo_img = resample_to_img(yeo7_img, zmap_img, interpolation='nearest')
buckner_img = resample_to_img(load_img(buckner_path), zmap_img, interpolation='nearest')
cereb_img = resample_to_img(load_img(cereb_path), zmap_img, interpolation='nearest')

# === Extract data arrays ===
cort_data = cort_img.get_fdata()
subcort_data = sub_img.get_fdata()
yeo_data = yeo_img.get_fdata()
buckner_data = buckner_img.get_fdata()
cereb_data = cereb_img.get_fdata()

# === Combine cortical and subcortical labels ===
combined_labels = {i: l for i, l in enumerate(cortical.labels)}
combined_labels.update({i + 100: l for i, l in enumerate(subcortical.labels)})

# === Refine Yeo labels only for unknowns (0s) ===
def refine_yeo(voxel_vals):
    center = voxel_vals[len(voxel_vals) // 2]
    if center != 0:
        return center
    neighbors = [v for v in voxel_vals if v > 0]
    if neighbors:
        values, counts = np.unique(neighbors, return_counts=True)
        return values[np.argmax(counts)]
    return 0

yeo_data_refined = generic_filter(yeo_data, function=refine_yeo, size=3, mode='constant', cval=0)

# === Extract voxel-level info ===
cluster_voxels = np.argwhere(labeled_array > 0)
z_vals = zmap_data[cluster_voxels[:, 0], cluster_voxels[:, 1], cluster_voxels[:, 2]]
cluster_ids = labeled_array[cluster_voxels[:, 0], cluster_voxels[:, 1], cluster_voxels[:, 2]]
mni_coords = nib.affines.apply_affine(zmap_affine, cluster_voxels)

anat_labels, yeo_labels = [], []

for voxel in cluster_voxels:
    i, j, k = voxel
    cort_val = int(cort_data[i, j, k])
    sub_val = int(subcort_data[i, j, k])
    cereb_val = int(cereb_data[i, j, k])
    yeo_val = 0

    anat_label = "Unknown"

    # Subcortical first
    if sub_val > 0 and sub_val not in [1, 2, 3, 12, 13, 14]: # Ignore cortical/non-GM indices
        if sub_val in combined_labels:
            anat_label = combined_labels[sub_val + 100]  # Offset for subcorticals
            yeo_val = 0

    # Then cortical
    if cort_val > 0 and cort_val in combined_labels:
        anat_label = combined_labels[cort_val]
        yeo_val = int(yeo_data_refined[i, j, k])

    # Then cerebellar
    if cereb_val > 0:
        anat_label = cereb_labels.get(cereb_val, "Unknown Cerebellar")
        buckner_val = int(buckner_data[i, j, k])


    anat_labels.append(anat_label)

    # Yeo network assignment
    if cereb_val > 0:
        yeo_label = buckner_labels.get(buckner_val, "Cerebellar Unknown")
    else:
        yeo7_dict = {
            1: "Visual", 2: "Somatomotor", 3: "Dorsal Attention",
            4: "Ventral Attention", 5: "Limbic", 6: "Frontoparietal", 7: "Default"
        }
        yeo_label = yeo7_dict.get(yeo_val, "Unknown")

    yeo_labels.append(yeo_label)

# === Build dataframe ===
df = pd.DataFrame({
    'i': cluster_voxels[:, 0],
    'j': cluster_voxels[:, 1],
    'k': cluster_voxels[:, 2],
    'x': mni_coords[:, 0],
    'y': mni_coords[:, 1],
    'z': mni_coords[:, 2],
    'Zvalue': z_vals,
    'ClusterID': cluster_ids,
    'AnatomicalRegion': anat_labels,
    'YeoNetwork': yeo_labels
})

# === Remove non-GM voxels ===
df = df[~df['AnatomicalRegion'].isin(["Background", "Unknown"])]

# === Add cluster stats ===
df['ClusterSize'] = df['ClusterID'].map(df.groupby('ClusterID').size())
df['ClusterPeakZ'] = df['ClusterID'].map(df.groupby('ClusterID')['Zvalue'].max())

# === Sort and export ===
df.sort_values(by=['ClusterID', 'z', 'y', 'x'], inplace=True)
df.to_csv("voxel_table_with_clusters_baseline.csv", index=False)
print("✅ Saved to voxel_table_with_clusters_baseline.csv")


✅ Saved to voxel_table_with_clusters_baseline.csv
